## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

In [ ]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT



### Load Translator resources


In [ ]:
APInames, metaKG, Translator_KP_info = translator_metakg.load_translator_resources()

In [ ]:
#Translator_KP_info

In [ ]:
All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

# generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

## Find the neighborhood of an entity from a subset of APIs 


In [ ]:
# select a list of APIs to use and a list of predicates to use, if the list is empty, use all APIs and predicates
#selected_APIlist = ['Microbiome KP - TRAPI 1.5.0']
selected_APIlist = []
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
print(select_APIs)
print(selected_metaKG.shape)


In [ ]:
name_resolver.lookup('acute myeloid leukemia', return_top_response=False, biolink_type='biolink:Disease',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results


In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [ ]:
#name_resolver.lookup('BCL2')
#name_resolver.lookup('BCL2', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [ ]:
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT.Neighborhood_finder('MONDO:0018874',
                                                                                            node2_categories = ['biolink:Disease'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

In [ ]:
# write a result to a json file
import json
with open('TCT_neighborhood_finder_result.json', 'w') as f:
    json.dump(result, f)

In [ ]:
# Step 8: Visualize the results
TCT.visulization_one_hop_ranking(result_ranked_by_primary_infores, result_parsed, 
                                num_of_nodes = 50, input_query = input_node_id, 
                                fontsize = 5)

In [ ]:
result_ranked_by_primary_infores

In [ ]:
from TCT import TCT_Visualization

dic_graph = TCT_Visualization.visualize_neighborhood_graph(result, show_label=True, height="500", width="100%")

In [ ]:
# End of the example
